In [1]:
from src.utils import load_config
from src.models.personalized_iap_packs import MODEL_REGISTRY
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

config = load_config("configs/personalized_iap_packs.yaml")


In [2]:
processed_data_path = config['processed_data_path']
if not os.path.exists(processed_data_path):
    print(f"❌ HATA: Gerekli bir dosya bulunamadı -> {processed_data_path}")

In [3]:
df = pd.read_excel(processed_data_path)

In [4]:
df.describe().T


,count,mean,std,min,25%,50%,75%,max
age,4643.0,39.462201,9.285158,18.0,35.0,39.0,43.0,9.900000e+01
total_spent_in_usd,4643.0,2.495328,7.031745,0.0,0.0,0.0,0.0,8.430002e+01
total_spent_time,4643.0,0.563429,1.683106,0.0,0.0,0.0,0.0,2.300000e+01
session_count,4643.0,29.029507,53.333551,1.0,3.0,9.0,29.5,6.140000e+02
total_session_time,4643.0,47155.449278,110343.073164,0.0,2520.0,10603.0,43982.0,1.699008e+06
easy_completion_rate,4643.0,93.937110,16.323090,50.0,100.0,100.0,100.0,1.000000e+02
medium_completion_rate,4643.0,53.542968,38.084715,0.0,25.0,50.0,100.0,1.000000e+02
hard_completion_rate,4643.0,5.537907,14.068065,0.0,0.0,0.0,0.0,1.000000e+02


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4643 entries, 0 to 4642
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   user_key_id             4643 non-null   object 
 1   country                 4643 non-null   object 
 2   age                     4643 non-null   int64  
 3   sex                     4643 non-null   object 
 4   total_spent_in_usd      4643 non-null   float64
 5   total_spent_time        4643 non-null   int64  
 6   session_count           4643 non-null   int64  
 7   total_session_time      4643 non-null   int64  
 8   easy_completion_rate    4643 non-null   int64  
 9   medium_completion_rate  4643 non-null   int64  
 10  hard_completion_rate    4643 non-null   float64
dtypes: float64(2), int64(6), object(3)
memory usage: 399.1+ KB


In [7]:
import mlflow
import mlflow.sklearn
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Make sure this path exists or create it
output_dir = "./mlruns/iap_packs"
os.makedirs(output_dir, exist_ok=True)

label_encoders = {}
for col in ['country', 'sex']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

    label_encoders[col] = list(le.classes_)

# Scale features once
features = [
    'country', 'age', 'sex', 'total_spent_in_usd', 'total_spent_time', 'session_count',
    'total_session_time', 'easy_completion_rate',
    'medium_completion_rate', 'hard_completion_rate'
]
scaler = StandardScaler()
features_scaled = scaler.fit_transform(df[features])

mlflow.set_experiment(config['experiment_name'])

for model_config in config['models']:
    model_name = model_config['model_name']
    ModelClass = MODEL_REGISTRY[model_name]

    for params in model_config['param_grid']:
        with mlflow.start_run(run_name=f"{model_name}"):

            # Log model name and parameters
            mlflow.log_param("model_name", model_name)
            mlflow.log_params(params)

            model = ModelClass(params)

            df['cluster'] = model.fit_predict(features_scaled)

            
            cluster_label_map = {
                0: 'Cheap',
                1: 'Fair',
                2: 'Expensive',
            }
            df['user_iap_pack_price_category'] = df['cluster'].map(cluster_label_map)

            # Save result to CSV
            result_path = os.path.join(output_dir, f"{model_name}_{str(params)}_results.csv".replace(" ", "_").replace(":", ""))
            df[['user_key_id', 'cluster', 'user_iap_pack_price_category']].to_csv(result_path, index=False)
            mlflow.log_artifact(result_path)

            labels = df['cluster']
            num_clusters = labels.nunique()

            if num_clusters >= 3:
                try:
                    sil = silhouette_score(features_scaled, labels)
                    ch = calinski_harabasz_score(features_scaled, labels)
                    db = davies_bouldin_score(features_scaled, labels)
                    
                    mlflow.log_metric("silhouette_score", sil)
                    mlflow.log_metric("calinski_harabasz_score", ch)
                    mlflow.log_metric("davies_bouldin_score", db)
                except:
                    print(f"Metric logging failed: {e}")
                    
                # Log individual cluster sizes
                cluster_counts = df['cluster'].value_counts().to_dict()
                for cluster_id, count in cluster_counts.items():
                    mlflow.log_metric(f"cluster_{cluster_id}_size", count)

            try:
                mlflow.sklearn.log_model(model.model, name="model")
            
                mlflow.log_dict(label_encoders, "iap_packs_label_encoders.json")

                run_info = {
                    "training_columns": list(features)
                }
                mlflow.log_dict(run_info, "iap_packs_run_info.json")
            except Exception as e:
                print(f"Model logging failed: {e}")

            print(f"Logged {model_name} with params {params}")


Traceback (most recent call last):
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 347, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 445, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 1588, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3.13/site-packages/mlflow/store/tracking/file_store.py", line 1581, in _read_helper
    result = read_yaml(root, file_name)
  File "/Users/emirhanbaran/Desktop/ai-based-game-platform-v1/venv/lib/python3

Logged KMeansClusterer with params {'n_clusters': 3, 'init': 'k-means++', 'n_init': 10, 'max_iter': 300}


2025/06/21 22:02:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged KMeansClusterer with params {'n_clusters': 5, 'init': 'k-means++', 'n_init': 10, 'max_iter': 300}


2025/06/21 22:02:56 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


...

Logged KMeansClusterer with params {'n_clusters': 3, 'init': 'k-means++', 'n_init': 10, 'max_iter': 1000}


2025/06/21 22:02:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged KMeansClusterer with params {'n_clusters': 5, 'init': 'k-means++', 'n_init': 10, 'max_iter': 300}


2025/06/21 22:03:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged KMeansClusterer with params {'n_clusters': 7, 'init': 'k-means++', 'n_init': 10, 'max_iter': 1000}
